# 05.4 — Learning rate scheduling

The learning rate is the most important hyperparameter ([02.7](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb)), and the best value *changes over training*: large early for fast progress, small late for fine convergence. **Scheduling** adjusts it on a policy. This notebook covers PyTorch's schedulers AND the honest reality of how *this* codebase does LR control — which turns out to be the mechanism underneath every scheduler, driven by the curriculum.

This notebook covers:

* The general scheduler idea and PyTorch's `lr_scheduler` family.
* The one mechanism they all share: setting `param_group["lr"]`.
* How this project controls LR — direct param-group manipulation via the freeze curriculum, not a scheduler object.
* The MATLAB piecewise-schedule config fields and where they live.

**Prerequisites:** [02.7 optimizers](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb), [05.1 the loop](05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

MATLAB's piecewise schedule drops the LR by a factor every N epochs:

```matlab
'InitialLearnRate', 1e-3, ...
'LearnRateSchedule', 'piecewise', ...
'LearnRateDropFactor', 0.9, ...
'LearnRateDropPeriod', 30, ...
```

This project's `base.yaml` carries the same three numbers — `initial_learning_rate: 0.001`, `learning_rate_decay: 0.9`, `learning_rate_epoch_drop: 30` — plus `learning_rate_epoch_ramp` for warmup. But how the Python side *applies* LR change is the interesting part, and it isn't a drop-in scheduler.

## Section 2 — The Python concepts you need

### 2.1 — The one mechanism: `param_group["lr"]`

Every LR change in PyTorch — whether from a fancy scheduler or by hand — ultimately writes to `optimizer.param_groups[i]["lr"]`. The optimizer reads that value each `step()`. That's the whole mechanism; schedulers are just policies for *what* to write:

In [ ]:
import torch
from torch import nn

model = nn.Linear(4, 2)
opt = torch.optim.Adam(model.parameters(), lr=0.1)
print("lr the optimizer will use:", opt.param_groups[0]["lr"])

opt.param_groups[0]["lr"] = 0.05          # set it directly — no scheduler needed
print("after manual change:      ", opt.param_groups[0]["lr"])

`param_groups` is a list because an optimizer can manage several groups with independent LRs ([02.7 §2.4](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb)) — which is exactly what this project exploits (§2.4). Writing `["lr"]` is all a scheduler does under its abstraction.

### 2.2 — PyTorch's scheduler family

For standard policies, `torch.optim.lr_scheduler` wraps the optimizer and writes the LR for you on `.step()`. The MATLAB piecewise schedule maps to `StepLR`:

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# MATLAB: DropFactor=0.9, DropPeriod=30  ↔  StepLR(gamma=0.9, step_size=30)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.9)

lrs = []
for epoch in range(90):
    opt.step()                 # (a real loop trains here first)
    scheduler.step()           # advance the schedule — once per EPOCH
    lrs.append(scheduler.get_last_lr()[0])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lrs)
ax.set_xlabel("epoch"); ax.set_ylabel("learning rate")
ax.set_title("StepLR: ×0.9 every 30 epochs (MATLAB piecewise, DropFactor=0.9, DropPeriod=30)")
plt.tight_layout(); plt.show()
print(f"lr: {lrs[0]:.2e} (start) → {lrs[29]:.2e} (ep 29) → {lrs[30]:.2e} (ep 30, dropped) → {lrs[-1]:.2e} (end)")

The two visible drops (at epochs 30 and 60) are the piecewise steps. Other schedulers exist — `CosineAnnealingLR`, `ReduceLROnPlateau`, `OneCycleLR` — but all reduce to §2.1: writing `param_group["lr"]`.

### 2.3 — How THIS project actually controls LR

Here's the honest reality, and it's a good lesson in reading a codebase rather than assuming. The project does **not** use a `torch.optim.lr_scheduler` object. Instead, the **freeze curriculum** ([Module 07](../README.md)) writes each param-group's LR directly every epoch — `param_group["lr"] = base_lr × freeze_factor` — via `apply_freeze_to_optimizer`. A freeze factor of 0 "soft-freezes" a submodule (LR → 0) while keeping optimizer momentum alive; a factor of 1 trains it at full rate. Scheduling and freezing are the *same operation* here.

In [ ]:
from neural_data_decoding.training.freezing import build_optimizer_with_module_groups

# Build an optimizer with named per-module groups (the encoder + classifier):
enc = nn.Linear(8, 16); clf = nn.Linear(16, 3)
opt = build_optimizer_with_module_groups(
    module_groups={"encoder": enc, "classifier": clf},
    initial_lr=1e-3, weight_decay=0.0,
)
for g in opt.param_groups:
    print(f"  group '{g['name']}': lr = {g['lr']}")

# The curriculum would then set each group's lr = base_lr × freeze_factor per epoch.
# Simulate a 'soft freeze' of the encoder (factor 0) while classifier trains (factor 1):
for g in opt.param_groups:
    factor = 0.0 if g["name"] == "encoder" else 1.0
    g["lr"] = 1e-3 * factor
print("after a soft-freeze of the encoder:")
for g in opt.param_groups:
    print(f"  group '{g['name']}': lr = {g['lr']}  {'(frozen)' if g['lr']==0 else '(training)'}")

### 2.4 — Why per-group, not a global scheduler

Because the curriculum thaws the encoder, decoder, and classifier on *different* schedules, a single global LR can't express it — each submodule needs its own LR trajectory. That's exactly the per-group `param_groups` mechanism from §2.1/§2.3. The config's `learning_rate_decay`/`learning_rate_epoch_drop` fields are carried for parity and future global-decay use, but the *live* LR control in the Optimal run is the freeze curriculum's per-group factors. **Read the code, not the config field names** — the same lesson as [03.5](../03_data_pipeline/03.5_normalization_recipes.ipynb)'s live-vs-stub recipes.

## Section 3 — The `neural_data_decoding` implementation

`apply_freeze_to_optimizer` — the per-group LR write that IS the project's scheduler:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/freezing.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip().startswith("for group in optimizer.param_groups"))
for k in range(i - 2, min(i + 7, len(src))):
    marker = " ►" if 'group["lr"]' in src[k] else "  "
    print(f"{k + 1:4}{marker}| {src[k]}")

Things to spot:

* `group["lr"] = base_lr * factor` — §2.1's mechanism, in production. The factor comes from the freeze schedule, read live per epoch.
* Groups whose names aren't in the schedule keep their current LR — so a partially-specified curriculum only touches the modules it names.
* The docstring cites `cgg_setFrozenNetwork_v2` and MATLAB's `setLearnRateFactor` — the MATLAB mechanism this mirrors. MATLAB *also* does LR-factor-per-layer rather than a monolithic schedule; the port matches that design.
* No scheduler object anywhere — the curriculum ([07.5](../README.md)) drives everything through this one function.

## Section 4 — Hands-on exercises

### Exercise 4.1 — build StepLR by hand

Reproduce `StepLR(step_size=10, gamma=0.5)` for 25 epochs using ONLY `param_group["lr"] = ...` (no scheduler object). Match the values PyTorch would produce.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
base = 0.1
by_hand = []
for epoch in range(25):
    opt.param_groups[0]["lr"] = base * (0.5 ** (epoch // 10))    # StepLR formula
    by_hand.append(opt.param_groups[0]["lr"])
print("epochs 0, 9, 10, 20:", [round(by_hand[e], 5) for e in (0, 9, 10, 20)])
print("→ 0.1, 0.1, 0.05, 0.025 — exactly StepLR(step=10, gamma=0.5)")

### Exercise 4.2 — independent per-group LRs

Give a two-group optimizer different LR trajectories: group A constant, group B halving each epoch. Print both LRs for 4 epochs.

In [ ]:
# Reveal:
a = nn.Linear(4, 4); b = nn.Linear(4, 4)
opt = torch.optim.Adam([{"params": a.parameters(), "lr": 0.1, "name": "A"},
                        {"params": b.parameters(), "lr": 0.1, "name": "B"}])
for epoch in range(4):
    opt.param_groups[1]["lr"] = 0.1 * (0.5 ** epoch)     # only group B decays
    print(f"epoch {epoch}: A lr={opt.param_groups[0]['lr']:.4f}, B lr={opt.param_groups[1]['lr']:.4f}")

## Section 5 — Common errors

### Scheduler seems to do nothing

`scheduler.step()` not called (or called before `optimizer.step()`). Order per epoch: `optimizer.step()` (per batch) then `scheduler.step()` (per epoch). PyTorch warns if you invert them.

### LR changes but training doesn't respond

You wrote to a copy, not the live `param_groups`. `optimizer.param_groups[i]["lr"] = x` is the real handle; a scheduler built over a *different* optimizer instance won't touch this one.

### Expecting a scheduler that isn't there

This project has no `lr_scheduler` object — LR is the curriculum's per-group factors (§2.3). Searching for `StepLR` in the codebase finds nothing; searching for `param_group["lr"]` finds the real mechanism.

### Rebuilt the optimizer, LR schedule reset

A scheduler holds a reference to a specific optimizer and an internal epoch counter — rebuild the optimizer ([02.7 §5](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb)) and the scheduler is orphaned. Reconstruct both together.

### Warmup expected but missing

`learning_rate_epoch_ramp` in the config would ramp the LR up over the first epochs (avoiding the instability of a cold large LR). Confirm it's non-zero and wired — like the decay fields, it's carried in config but the live behavior is curriculum-driven.

## Section 6 — Further reading

- [lr_scheduler docs](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate) — every built-in scheduler.
- [How to pick a learning rate schedule](https://www.deeplearning.ai/ai-notes/optimization/) — the intuition behind decay + warmup.
- [`src/neural_data_decoding/training/freezing.py`](../../src/neural_data_decoding/training/freezing.py) — `apply_freeze_to_optimizer`, the project's real LR control.

Next up: **[05.5 checkpoint & resume state machine](05.5_checkpoint_resume_state_machine.ipynb)** — saving, resuming, and the Current-vs-Optimal distinction.